In [ ]:
import importlib
import tfr_stats as tfs
importlib.reload(tfs)                                  

COMPARE_CONDITIONS = [("late", "short"), ("correct", "long")]  

N_PERM = 1000        # number of label shuffles 
PVAL   = 0.05        # two-tailed threshold
DECIM  = 1           # keep every 10th time sample (faster + less memory)
SEED   = 0           # fixed seed so same result every run
BAND_RANGES = CONFIG["band_ranges"]                    # <- the dict {band: (fmin, fmax)} you already use for the band lines;
                                                       #    change the name if yours is called differently

# %% STEP 1 - single-trial power + common baseline (SLOW part, run it once) ----------
prep = tfs.prepare_trial_data(
    ds_data, CONFIG["session_dir"], CONFIG["behavior_csv"], COMPARE_CONDITIONS,
    CONFIG["channel_id"], CONFIG["events"], CONFIG["band_settings"],
    CONFIG["tmin"], CONFIG["tmax"], fs,
    CONFIG["baseline_window"], CONFIG["crop_window"], CONFIG["baseline_event"],
    decim=DECIM)
# CHECK: "Overlap check OK" is printed; the n per condition and event are what you expect
#        (they can differ between Press / Tone / Release because some trials have no valid event time).

# %% STEP 2 - look at the data before testing ---------------------------------------
import numpy as np
for ev, (a, b) in prep["data"].items():
    print(ev, "A:", a.shape, "B:", b.shape,
          "| all finite:", np.isfinite(a).all() and np.isfinite(b).all(),
          "| mean log-ratio A/B: %.3f / %.3f" % (a.mean(), b.mean()))
print("time axis:", prep["times"][0], "->", prep["times"][-1], "s,", len(prep["times"]), "samples")
print("freq axis:", prep["freqs"][0], "->", prep["freqs"][-1], "Hz,", len(prep["freqs"]), "freqs",
      "| duplicated frequencies:", len(prep["freqs"]) - len(np.unique(prep["freqs"])))
# CHECK: all finite = True; log-ratio means are small numbers (about -0.5 ... +0.5);
#        duplicated frequencies = 0 (if not: neighbouring bands share an edge frequency -> fix band_settings).

# %% STEP 3 - sanity checks of the shuffling and the test ----------------------------
first_event = list(prep["data"].keys())[0]
a, b = prep["data"][first_event]
tfs.check_shuffle_logic(a, b)                          # group sizes unchanged, means move closer together
tfs.false_positive_check(a, n_perm=200, pval=PVAL)     # random split of ONE condition: nothing should be significant
# CHECK: uncorrected ~ 5% (anything between ~2% and ~10% is fine because neighbouring pixels are correlated),
#        corrected ~ 0%. If corrected is clearly above 0 something is wrong -> tell me before going on.

# %% STEP 4 - quick test run (100 permutations), then the real run -------------------
stats, bands = tfs.run_permutation_stats(prep, BAND_RANGES, n_perm=100, pval=PVAL, seed=SEED)   # just to see that it works
tfs.print_stats_summary(stats)

stats, bands = tfs.run_permutation_stats(prep, BAND_RANGES, n_perm=N_PERM, pval=PVAL, seed=SEED)  # the real run
tfs.print_stats_summary(stats)

# %% STEP 5 - plots --------------------------------------------------------------------
labels = prep["labels"]
n_txt = " | ".join(f"{ev}: nA={n[0]}, nB={n[1]}" for ev, n in prep["n_valid"].items())
title = f"Permutation test: {labels[0]} vs {labels[1]} - {CONFIG['channel_id']} ({N_PERM} permutations; {n_txt})"
fname = ("perm_" + "_vs_".join(f"{o}-{t}" for o, t in COMPARE_CONDITIONS) + f"_{CONFIG['channel_id']}")

tfs.plot_permutation_maps(                                       # full time-frequency maps
    stats, prep["times"], prep["freqs"], event_names, labels, title,
    freq_range=CONFIG["full_freq_range"], yticks=CONFIG["full_yticks"],
    band_boundaries=CONFIG["oscillation_boundaries"],
    save_path=tfx.make_save_path_named(CONFIG, fname + "_maps"))

tfs.plot_permutation_bands(                                      # one row per band, lines of both conditions
    stats, prep["times"], event_names, bands, labels, title,
    save_path=tfx.make_save_path_named(CONFIG, fname + "_bands"))